# Status: nonlinear simulation under development
The supplied run stopped with `ArgumentError: No matching component found in allowedcomponents.`
Cleared outputs do not imply a successful run. The Python parameter workflow is the validated entry point.
No amplifier gain, layout readiness or fabrication performance is claimed.


# SNAIL JTWPA — JosephsonCircuits.jl simulation

This notebook takes the validated Python parameter handoff and performs the
nonlinear circuit simulation in stages.

### Simulation ladder

**1.** package/version sanity check  
**2.** one SNAIL + DC bias calibration  
**3.** one SNAIL small-signal check  
**4.** 5–20-cell unpumped line  
**5.** 5–20-cell pumped 3WM  
**6.** pump-current sweep  
**7.** 50 → 100 → 300 → 600 cells  
**8.** export the physical netlist for SPICE/layout/EM

Do not jump directly to a 600-cell nonlinear solve.

In [ ]:
using Pkg
# Pkg.add("JosephsonCircuits")
# Pkg.add("Plots")
# Pkg.add("DataFrames")
# Pkg.add("CSV")

using JosephsonCircuits
using Plots
using DataFrames
using CSV
using Printf

println("Julia version: ",VERSION)
println("Threads: ",Threads.nthreads())

## 1. Baseline design parameters

Starting point from Python:

- `n = 3`
- `alpha ≈ 0.29`
- `Phi_ext/Phi0 ≈ 0.4089`
- `Lj_large ≈ 41.1 pH`
- `Ic_large ≈ 8.0 µA`
- `Ic_small ≈ 2.32 µA`
- `Cshunt ≈ 58 fF`
- `Lloop ≈ 0.74 pH`
- pump = 12 GHz
- signal = 4–8 GHz

In [ ]:
const Phi0 = JosephsonCircuits.Phi0
const phi0 = JosephsonCircuits.phi0

n_large = 3
alpha = 0.29
flux_ext_over_phi0 = 0.408912098312476

Z0 = 50.0
Lcell = 145e-12
Cshunt = 58e-15

Lj_large = 41.1449301218e-12
Ic_large = 7.9987006297e-6
Ic_small = alpha*Ic_large

Cj_large = 35.9941528338e-15
Cj_small = 10.4383043218e-15

Lloop = 0.74e-12

M_bias = 5.0e-12
Idc = flux_ext_over_phi0*Phi0/M_bias

fp = 12e9
fs_start, fs_stop = 4e9, 8e9

N_cells_small = 10
N_cells_medium = 50
N_cells_final = 600

@printf("Idc ≈ %.3f µA\n",Idc*1e6)
@printf("Ic_large = %.3f µA\n",Ic_large*1e6)
@printf("Ic_small = %.3f µA\n",Ic_small*1e6)

In [ ]:
Lbias_value = 0.70e-9 # H = 0.70 nH
Kbias_value = M_bias / sqrt(Lloop * Lbias_value)

## 2. Physical SNAIL cell

The RF path is:

```text
        ┌── JJ_L ─ JJ_L ─ JJ_L ─┐
left ───┤                        ├── right
        └── Lloop ───────────────┘
        └────── JJ_small ────────┘

left ── Cshunt ── ground
```

The small junction has `Ic_small = alpha*Ic_large`.

A small linear loop inductance is retained so that a separate DC bias coil can
be coupled to the SNAIL loop.

The RF pump is applied to the RF input port, **not** to the DC bias port.

In [ ]:

@variables Rport Cg Cj Lj Lloop Lbias Kbias

function add_snail_cell!(circuit,left::Int,right::Int;suffix::String)
    # Small JJ directly across the RF terminals
    push!(circuit,("Js_$suffix","$(left)","$(right)",Lj/alpha))
    push!(circuit,("Cs_$suffix","$(left)","$(right)",Cj/alpha))

    # Large-JJ branch: left -> Lloop -> JJ1 -> JJ2 -> JJ3 -> right
    n1 = right+1
    n2 = right+2
    n3 = right+3
    n4 = right+4

    push!(circuit,("Lg_$suffix","$(left)","$(n1)",Lloop))
    push!(circuit,("J1_$suffix","$(n1)","$(n2)",Lj))
    push!(circuit,("C1_$suffix","$(n1)","$(n2)",Cj))
    push!(circuit,("J2_$suffix","$(n2)","$(n3)",Lj))
    push!(circuit,("C2_$suffix","$(n2)","$(n3)",Cj))
    push!(circuit,("J3_$suffix","$(n3)","$(n4)",Lj))
    push!(circuit,("C3_$suffix","$(n3)","$(n4)",Cj))

    # Numerically negligible connection from the last JJ to the RF node.
    push!(circuit,("wire_$suffix","$(n4)","$(right)",1e-18))

    # One segment of the distributed DC bias line.
    # The same DC current can flow through all segments in series.
    b1 = n4+1
    b2 = n4+2
    push!(circuit,("Lb_$suffix","$(b1)","$(b2)",Lbias))

    # Mutual coupling from the bias-line segment to the SNAIL loop inductance.
    push!(circuit,("Kb_$suffix","Lg_$suffix","Lb_$suffix",Kbias))

    return (nret=n4+2,b1=b1,b2=b2)
end


### Bias-coil starting values

The mutual inductance is initially treated as an engineering parameter. The
layout/EM model must later extract it.

For example,

\[
\Phi_\mathrm{ext}=MI_\mathrm{bias}.
\]

The bias coil is **not** used as the microwave pump path.

In [ ]:
# Mutual bias parameters
Lloop = 0.74e-12      # H = 0.74 pH
M_bias = 5.0e-12      # H = 5 pH
Lbias_value = 0.70e-9 # H = 0.70 nH

Kbias_value = M_bias / sqrt(Lloop * Lbias_value)

@printf("Lloop = %.3f pH\n", Lloop * 1e12)
@printf("M_bias = %.3f pH\n", M_bias * 1e12)
@printf("Lbias = %.3f nH\n", Lbias_value * 1e9)
@printf("Kbias = %.4f\n", Kbias_value)

@assert Kbias_value < 1.0 "Invalid mutual coupling: K >= 1"

## 3. Build a small-N SNAIL line

For the first line-level validation we use a uniform chain:

- nonlinear SNAIL per cell,
- one shunt capacitor per cell,
- 50-ohm input/output,
- no periodic resonant phase matching,
- no 2`fp` stopband.

In [ ]:

function build_snail_line(Ncells::Int)
    circuit = Tuple{String,String,String,Any}[]

    # RF input
    push!(circuit,("P_in","1","0",1))
    push!(circuit,("R_in","1","0",50.0))

    left = 1
    next_free = 2

    bias_b1 = 0
    previous_bias_b2 = 0

    for i in 1:Ncells
        right = next_free

        # RF shunt capacitor
        Cval = (i == 1 || i == Ncells) ? Cshunt/2 : Cshunt
        push!(circuit,("Cg_$i","$(left)","0",Cval))

        ret = add_snail_cell!(
            circuit,left,right;
            suffix="cell$(i)"
        )

        # Connect the distributed bias-line segments in series.
        if i == 1
            bias_b1 = ret.b1
        else
            push!(circuit,("bias_wire_$i","$(previous_bias_b2)","$(ret.b1)",1e-18))
        end
        previous_bias_b2 = ret.b2

        next_free = ret.nret+1
        left = right
    end

    # RF output
    push!(circuit,("Cg_out","$(left)","0",Cshunt/2))
    push!(circuit,("R_out","$(left)","0",50.0))
    push!(circuit,("P_out","$(left)","0",2))

    # One DC-bias port drives the complete distributed bias line.
    # The 1 kOhm resistor gives the bias source a finite DC return.
    push!(circuit,("P_bias","$(bias_b1)","0",3))
    push!(circuit,("R_bias","$(bias_b1)","0",1000.0))

    # Complete the bias-line DC path.
    push!(circuit,("bias_return","$(previous_bias_b2)","0",1e-18))

    return circuit
end


## 4. Important DC-bias implementation note

The explicit bias coil is useful for validating the physical SNAIL topology, but
a 600-cell model with hundreds of independently driven bias ports is not a good
first calculation.

Use the progression:

`1 → 5 → 10 → 20 → 50 → 100 → 300 → 600`.

At each stage, verify that the RF model agrees with the previous stage.

If the explicit bias network becomes the dominant source of RF loading, replace
it with the extracted distributed bias network from the layout/EM model.

Do not repair convergence by inserting arbitrary large grounding inductors into
the SNAIL loop.

In [ ]:
# Small-N circuit construction
circuit_small = build_snail_line(N_cells_small)

println("Small-N component count = ",length(circuit_small))

# Define the RF component values.
# The bias branch is included in each cell; the exact source topology should
# be calibrated using the installed JosephsonCircuits version.
circuitdefs_small = Dict(
    Cg => Cshunt,
    Cj => Cj_large,
    Lj => Lj_large,
    Lloop => Lloop,
    Lbias => Lbias_value,
    Kbias => Kbias_value,
)


# Small-N circuit construction
circuit_small = build_snail_line(N_cells_small)

println("Small-N component count = ",length(circuit_small))

circuitdefs_small = Dict(
    Cg => Cshunt,
    Cj => Cj_large,
    Lj => Lj_large,
    Lloop => Lloop,
    Lbias => Lbias_value,
    Kbias => Kbias_value,
)


In [ ]:
# Frequency grid for the first linear check
ws = 2*pi*(3.0:0.05:14.0)*1e9
wp = (2*pi*fp,)

Npumpharmonics = (1,)
Nmodulationharmonics = (1,)

# Source skeleton:
#
sources_dc = [
    (mode=(0,),port=3,current=Idc),
]

sol_dc = hbsolve(
    ws,wp,sources_dc,
    Nmodulationharmonics,Npumpharmonics,
    circuit_small,circuitdefs_small;
    dc=true,
    threewavemixing=true,
    fourwavemixing=true,
)
#
# The exact bias-port number depends on how the final bias network is wired.
# Confirm it with the actual generated circuit before running.

In [ ]:

# DC-only operating-point test
ws = 2*pi*(3.0:0.05:14.0)*1e9
wp = (2*pi*fp,)

Npumpharmonics = (1,)
Nmodulationharmonics = (1,)

sources_dc = [
    (mode=(0,),port=3,current=Idc),
]

# Run after confirming the circuit parses:
#
# sol_dc = hbsolve(
#     ws,wp,sources_dc,
#     Nmodulationharmonics,Npumpharmonics,
#     circuit_small,circuitdefs_small;
#     dc=true,
#     threewavemixing=true,
#     fourwavemixing=true,
# )
#
# plot(
#     ws/(2*pi*1e9),
#     10*log10.(abs2.(sol_dc.linearized.S(
#         outputmode=(0,),outputport=2,
#         inputmode=(0,),inputport=1,
#         freqindex=:
#     ))),
#     xlabel="Frequency (GHz)",
#     ylabel="S21 (dB)",
#     title="Small-N unpumped SNAIL line"
# )


In [ ]:
pump_fractions = [0.03,0.05,0.075,0.10,0.15,0.20]
Ip_values = pump_fractions .* Ic_large

ws_pump = 2*pi*(4.0:0.05:8.0)*1e9
wp = (2*pi*fp,)

Npumpharmonics = (8,)
Nmodulationharmonics = (4,)

# Example execution skeleton after the bias port is confirmed:
#
# results = Dict()
# for Ip in Ip_values
#     sources = [
#         (mode=(0,),port=3,current=Idc),
#         (mode=(1,),port=1,current=Ip),
#     ]

#     sol = hbsolve(
#         ws_pump,wp,sources,
#         Nmodulationharmonics,Npumpharmonics,
#         circuit_small,circuitdefs_small;
#         dc=true,
#         threewavemixing=true,
#         fourwavemixing=true,
#         iterations=200,
#         switchofflinesearchtol=0.0,
#         alphamin=1e-7,
#     )
#     results[Ip] = sol
# end

## 7. Gain, reflection and idler diagnostics

For the two-port RF line,

\[
G=|S_{21}|^2.
\]

A valid 3WM amplifier should also show the expected idler at

\[
f_i=f_p-f_s.
\]

Inspect:

- signal `S21`,
- `S11`,
- reverse gain `S12`,
- idler conversion,
- commutation-relation error,
- quantum efficiency if available.

Do not accept a large positive `S21` alone as evidence of a correct JTWPA.

In [ ]:

pump_fractions = [0.03,0.05,0.075,0.10,0.15,0.20]
Ip_values = pump_fractions .* Ic_large

ws_pump = 2*pi*(4.0:0.05:8.0)*1e9
wp = (2*pi*fp,)

Npumpharmonics = (8,)
Nmodulationharmonics = (4,)

# Example execution:
#
# results = Dict()
# for Ip in Ip_values
#     sources = [
#         (mode=(0,),port=3,current=Idc),   # DC bias line
#         (mode=(1,),port=1,current=Ip),    # RF pump
#     ]
#
#     sol = hbsolve(
#         ws_pump,wp,sources,
#         Nmodulationharmonics,Npumpharmonics,
#         circuit_small,circuitdefs_small;
#         dc=true,
#         threewavemixing=true,
#         fourwavemixing=true,
#         iterations=200,
#         switchofflinesearchtol=0.0,
#         alphamin=1e-7,
#     )
#     results[Ip] = sol
# end


## 8. Compare Julia's cold line with the Python Bloch result

For the uniform RF ladder,

\[
\cos(k)=1-\frac{\omega^2LC}{2}.
\]

The Python notebook gives the reference curve. The unpumped small-N
JosephsonCircuits simulation should reproduce it away from the internal
junction modes.

This is a crucial cross-check before nonlinear gain is trusted.

In [ ]:
function bloch_k(f;L=Lcell,C=Cshunt)
    arg = 1 .- (2*pi .* f).^2 .* L .* C ./ 2
    return acos.(clamp.(arg,-1.0,1.0))
end

fs = range(fs_start,fs_stop,length=401)
fi = fp .- fs
dk_cold = bloch_k(fp) .- bloch_k(fs) .- bloch_k(fi)

p = plot(
    fs./1e9,dk_cold,
    xlabel="Signal frequency (GHz)",
    ylabel="Δk_cold (rad/cell)",
    label="cold",
    title="Cold 3WM mismatch"
)
hline!(p,[0.0],label=false)
display(p)

@printf("max |Δk| = %.4e rad/cell\n",maximum(abs.(dk_cold)))

## 9. Pump-induced phase mismatch

The cold mismatch is not the final criterion.

The relevant quantity is

\[
\Delta k_\mathrm{total}
=
k_p-k_s-k_i+\Delta k_\mathrm{NL}(I_p).
\]

The nonlinear term comes from the pump-shifted operating point. A practical
way to identify it is to compare the gain-length behavior for `N=10,20,50`
cells.

If the gain is exponential and smooth, the phase matching is likely good.

If gain oscillates strongly with length, phase mismatch is likely dominating.

## 10. Scale gradually

Recommended progression:

| Cells | Purpose |
|---:|---|
| 1 | SNAIL topology / DC bias |
| 5 | first nonlinear mixing |
| 10 | gain accumulation |
| 20 | phase mismatch |
| 50 | realistic gain trend |
| 100 | numerical scaling |
| 300 | near-final |
| 600 | final candidate |

Use low harmonic counts initially. Increase `Npumpharmonics` and
`Nmodulationharmonics` only after the lower-order calculation is converged.

In [ ]:
# Suggested settings
Ntest = 10
Npumpharmonics = (8,)
Nmodulationharmonics = (4,)

# Then:
# Ntest = 20
# Npumpharmonics = (12,)
# Nmodulationharmonics = (6,)

# Final convergence study:
# Npumpharmonics = (16,)
# Nmodulationharmonics = (8,)

## 11. Export the final physical circuit

After the small-N and pumped calculations pass, generate the final-size circuit
and export it.

The exported netlist can be used for SPICE cross-checks and as the reference
for the layout schematic.

In [ ]:
# Example after the final model is validated:
#
# final_circuit = build_snail_line(N_cells_final)
#
# final_defs = Dict(
#     Cg => Cshunt,
#     Cj => Cj_large,
#     Lj => Lj_large,
#     Lloop => Lloop,
#     Lbias => Lbias_value,
#     Kbias => Kbias_value,
# )
#
# JosephsonCircuits.export_netlist(
#     "snail_jtwpa_600cell.net",
#     final_circuit,
#     final_defs
# )
#
# println("Final netlist exported.")

# 12. Layout / EM handoff circuit

Each RF cell should contain:

```text
             ┌── JJ_L ─ JJ_L ─ JJ_L ─┐
RF node n ───┤                        ├── RF node n+1
             └── Lloop ───────────────┘
             └────── JJ_small ────────┘

RF node n ── Cshunt ── ground
```

The three large junctions are series-connected and the small junction is the
parallel branch.

A separate DC flux line is magnetically coupled to the SNAIL loop.

### Baseline electrical starting point

- `n = 3`
- `alpha = 0.29`
- `Phi_ext/Phi0 ≈ 0.409`
- `Ic_large ≈ 8.00 µA`
- `Ic_small ≈ 2.32 µA`
- `Lj_large ≈ 41.1 pH`
- provisional `Cj_large ≈ 36 fF`
- provisional `Cj_small ≈ 10.4 fF`
- `Cshunt ≈ 58 fF`
- `Lloop ≈ 0.74 pH`
- pump = 12 GHz
- signal = 4–8 GHz
- initial line = 600 cells

### EM must extract

1. actual SNAIL loop inductance,
2. mutual inductance to the flux line,
3. flux uniformity along the array,
4. shunt capacitor,
5. junction pad capacitance,
6. inter-cell stray capacitance,
7. ground inductance,
8. package/launch effects,
9. higher-order modes.

Return the extracted values to the Python notebook and repeat the parameter
optimization before tape-out.

# Final acceptance criteria

The design is ready for detailed EM only when:

- the single-SNAIL DC operating point is stable;
- simulated small-signal inductance agrees with Python;
- the unpumped 10–20-cell dispersion agrees with Python;
- signal/idler conversion occurs at `fi = fp - fs`;
- gain increases appropriately with cell count;
- nonlinear phase mismatch is acceptable;
- pump depletion is acceptable;
- no unwanted resonance sits in the operating band;
- junction phase excursions are acceptable;
- the bias network does not materially load the RF path;
- parameter tolerances have been checked.

Only then should the 600-cell result be treated as a serious layout candidate.